# Qwen/Qwen3-VL-2B-Instruct: Multi-Prompting Strategies Evaluation on Colab GPU
### Strategies: Zero-Shot Direct vs. One-Shot vs. Zero-Shot Chain-of-Thought (CoT)
**Requirements**: Make sure your Colab Runtime is set to **GPU** (`Runtime -> Change runtime type -> T4 GPU`).


In [ ]:
# 1. Install required packages
!pip install -q -U git+https://github.com/huggingface/transformers.git accelerate qwen-vl-utils


In [ ]:
import os
import sys
import json
import time
import gzip
import base64
import re
import torch
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float32

model_id = "Qwen/Qwen3-VL-2B-Instruct"
print(f"Loading {model_id} on {device}...")
processor = AutoProcessor.from_pretrained(model_id)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=dtype,
    device_map="auto" if device == "cuda" else None
)
model.eval()
print("Model loaded successfully!")


In [ ]:
# 3. Load Embedded Benchmark Data (100 Items: attr_relation, two_hop, spatial_conjunction)
import gzip, base64, json

BENCHMARK_B64 = """..."""

items = json.loads(gzip.decompress(base64.b64decode(BENCHMARK_B64.encode('ascii'))).decode('utf-8'))
print(f"Loaded {len(items)} benchmark items successfully.")
category_counts = {}
for it in items:
    tmpl = it["query"].get("template", "unknown")
    category_counts[tmpl] = category_counts.get(tmpl, 0) + 1
print("Category Breakdown:", category_counts)


In [ ]:
# 4. Prompt Generation & Extraction Functions
import re

def format_sg_components(sg):
    objects = sg.get("objects", [])
    relations = sg.get("relations", [])
    obj_lines = []
    for obj in objects:
        lbl = obj.get("label", "unknown")
        attrs = obj.get("attributes", [])
        attr_str = f"attributes: [{', '.join(attrs)}]" if attrs else "no attributes"
        obj_lines.append(f"  - {lbl} ({attr_str})")

    rel_lines = []
    for rel in relations:
        s = rel.get("subject", "")
        r = rel.get("relation", "")
        o = rel.get("object", "")
        rel_lines.append(f'  - "{s}" {r} "{o}"')
        
    return len(objects), "\n".join(obj_lines) if obj_lines else "  None", len(relations), "\n".join(rel_lines) if rel_lines else "  None"

def build_prompt_zero_shot_direct(item):
    num_obj, obj_str, num_rel, rel_str = format_sg_components(item)
    q = item["query"].get("question", "")
    return (
        "You are given a scene graph representation of an image containing objects and their relationships, followed by a question.\n\n"
        f"Scene Graph:\nObjects ({num_obj}):\n{obj_str}\n\n"
        f"Relationships ({num_rel}):\n{rel_str}\n\n"
        f"Question: {q}\n\n"
        "Instructions: Based strictly on the scene graph provided above, answer the question accurately. "
        "Provide only the direct concise answer (a few words or a single term) without explanation or reasoning.\n"
        "Answer:"
    )

def build_prompt_one_shot(item):
    num_obj, obj_str, num_rel, rel_str = format_sg_components(item)
    q = item["query"].get("question", "")
    return (
        "You are given a scene graph representation of an image containing objects and their relationships, followed by a question. Answer with the direct concise target entity.\n\n"
        "Example:\n"
        "Scene Graph:\n"
        "Objects (4):\n"
        "  - table (attributes: [wooden, brown])\n"
        "  - laptop (attributes: [silver])\n"
        "  - cup (attributes: [white, ceramic])\n"
        "  - notebook (attributes: [blue])\n\n"
        "Relationships (3):\n"
        '  - "laptop" on "table"\n'
        '  - "cup" to the right of "laptop"\n'
        '  - "notebook" to the right of "laptop"\n\n'
        "Question: What is on the table to the right of the laptop and to the left of the cup?\n"
        "Answer: notebook\n\n"
        "Now answer the question for the following scene graph:\n"
        f"Scene Graph:\nObjects ({num_obj}):\n{obj_str}\n\n"
        f"Relationships ({num_rel}):\n{rel_str}\n\n"
        f"Question: {q}\n"
        "Answer:"
    )

def build_prompt_zero_shot_cot(item):
    num_obj, obj_str, num_rel, rel_str = format_sg_components(item)
    q = item["query"].get("question", "")
    return (
        "You are given a scene graph representation of an image containing objects and their relationships, followed by a question.\n\n"
        f"Scene Graph:\nObjects ({num_obj}):\n{obj_str}\n\n"
        f"Relationships ({num_rel}):\n{rel_str}\n\n"
        f"Question: {q}\n\n"
        "Instructions:\n"
        "1. Reason step-by-step:\n"
        "   - Identify the key objects, attributes, or spatial landmarks mentioned in the question.\n"
        "   - Trace the specified relationship(s) or spatial arrangement across the graph. If multiple spatial constraints are given, identify the entity that satisfies all of them.\n"
        "2. Provide your final concise answer on the last line in the exact format:\n"
        "   Final Answer: <answer>"
    )

def extract_answer(pred_text, strategy_name):
    clean = pred_text.strip()
    if strategy_name == "zero_shot_cot":
        match = re.search(r"Final Answer:\s*(.+)", clean, re.IGNORECASE)
        if match:
            ans = match.group(1).strip()
            return ans.split("\n")[0].strip()
        lines = [line.strip() for line in clean.split("\n") if line.strip()]
        if lines:
            last = lines[-1]
            last = re.sub(r"^(?:answer|the answer is|final answer):\s*", "", last, flags=re.IGNORECASE)
            return last.strip()
    else:
        clean = re.sub(r"^(?:answer|the answer is|final answer):\s*", "", clean, flags=re.IGNORECASE).strip()
        first_line = clean.split("\n")[0].strip()
        return first_line.rstrip(".")
    return clean


In [ ]:
# 5. Run Evaluation Loop
import time, json
import pandas as pd

def evaluate_strategy(strategy_name, prompt_fn, max_tokens):
    print(f"\n{'='*60}\nRunning Strategy: {strategy_name} (max_tokens={max_tokens})\n{'='*60}")
    start = time.time()
    results = []
    correct, exact = 0, 0
    t_stats = {
        "attr_relation": {"total": 0, "correct": 0, "exact": 0},
        "two_hop": {"total": 0, "correct": 0, "exact": 0},
        "spatial_conjunction": {"total": 0, "correct": 0, "exact": 0}
    }

    for i, item in enumerate(items, 1):
        q_info = item["query"]
        gt = str(q_info.get("answer", "")).strip().lower()
        t = q_info.get("template", "unknown")
        if t not in t_stats:
            t_stats[t] = {"total": 0, "correct": 0, "exact": 0}
        t_stats[t]["total"] += 1

        prompt = prompt_fn(item)
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]

        inputs = processor.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt"
        )
        if device == "cuda":
            inputs = {k: v.to("cuda") for k, v in inputs.items()}

        with torch.no_grad():
            gen_kwargs = {
                "max_new_tokens": max_tokens,
                "do_sample": False,
                "repetition_penalty": 1.15
            }
            if strategy_name == "zero_shot_cot":
                gen_kwargs["no_repeat_ngram_size"] = 3
            out_ids = model.generate(**inputs, **gen_kwargs)
        
        gen_ids = [o[len(inp):] for inp, o in zip(inputs["input_ids"], out_ids)]
        raw_pred = processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()

        parsed = extract_answer(raw_pred, strategy_name)
        clean_pred = parsed.lower().rstrip(".").strip()

        is_exact = (clean_pred == gt)
        is_correct = is_exact or (gt in clean_pred) or (clean_pred in gt and len(clean_pred) > 1)

        if is_correct:
            correct += 1
            t_stats[t]["correct"] += 1
        if is_exact:
            exact += 1
            t_stats[t]["exact"] += 1

        results.append({
            "image_number": item["image_number"],
            "gqa_image_id": item.get("gqa_image_id"),
            "template": t,
            "question": q_info.get("question"),
            "ground_truth": gt,
            "raw_output": raw_pred,
            "parsed_answer": parsed,
            "is_correct": is_correct,
            "is_exact": is_exact
        })

        if i % 25 == 0 or i == len(items):
            print(f"[{i:03d}/{len(items)}] Accuracy: {correct}/{i} ({correct/i*100:.1f}%) | Elapsed: {time.time()-start:.1f}s")

    elapsed = time.time() - start
    summary = {
        "strategy": strategy_name,
        "total": len(items),
        "correct": correct,
        "exact": exact,
        "accuracy": round(correct / len(items), 4),
        "exact_accuracy": round(exact / len(items), 4),
        "time_seconds": round(elapsed, 2),
        "avg_time_per_sample": round(elapsed / len(items), 3),
        "category_breakdown": {
            k: {
                "total": v["total"],
                "correct": v["correct"],
                "exact": v["exact"],
                "accuracy": round(v["correct"]/v["total"], 4) if v["total"] else 0,
                "exact_accuracy": round(v["exact"]/v["total"], 4) if v["total"] else 0
            }
            for k, v in t_stats.items()
        }
    }
    
    with open(f"results_{strategy_name}.json", "w") as f:
        json.dump({"metadata": summary, "results": results}, f, indent=2)
        
    return summary

# Evaluate Direct (Primary, ~20s) and CoT (~60min)
strategies = [
    ("zero_shot_direct", build_prompt_zero_shot_direct, 32),
    ("one_shot", build_prompt_one_shot, 32)
    # To run Zero-Shot CoT, uncomment below:
    # ("zero_shot_cot", build_prompt_zero_shot_cot, 1024)
]

summaries = {}
for name, pfn, tok in strategies:
    summaries[name] = evaluate_strategy(name, pfn, tok)

with open("strategies_summary.json", "w") as f:
    json.dump(summaries, f, indent=2)

print("\n" + "="*70)
print("BENCHMARK COMPARISON SCOREBOARD:")
print("="*70)
df_rows = []
for s in summaries.values():
    bd = s.get("category_breakdown", {})
    df_rows.append({
        "Strategy": s["strategy"],
        "Overall Acc": f"{s['accuracy']*100:.1f}%",
        "Exact Match": f"{s.get('exact_accuracy', 0)*100:.1f}%",
        "attr_relation": f"{bd.get('attr_relation', {}).get('accuracy', 0)*100:.1f}%",
        "two_hop": f"{bd.get('two_hop', {}).get('accuracy', 0)*100:.1f}%",
        "spatial_conjunction": f"{bd.get('spatial_conjunction', {}).get('accuracy', 0)*100:.1f}%",
        "Time (s)": f"{s['time_seconds']:.1f}s"
    })

df = pd.DataFrame(df_rows)
print(df.to_string(index=False))
